# Exercice 4 — Classification MNIST avec un CNN

**Objectif :** construire un réseau de neurones convolutif (CNN) pour reconnaître les chiffres manuscrits de 0 à 9.

MNIST contient 60 000 images d'entraînement et 10 000 images de test, en niveaux de gris, de taille 28 × 28 pixels.

## 1. Chargement des données

In [ ]:
import torch
from torch import nn
from torch.utils.data import DataLoader
from torchvision import datasets, transforms
import matplotlib.pyplot as plt
import numpy as np

torch.manual_seed(42)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Appareil utilisé : {device}')

# Normalisation standard MNIST : mean=0.1307, std=0.3081
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.1307,), (0.3081,))
])

train_dataset = datasets.MNIST(root='./data', train=True,  download=True, transform=transform)
test_dataset  = datasets.MNIST(root='./data', train=False, download=True, transform=transform)

train_loader = DataLoader(train_dataset, batch_size=64,  shuffle=True)
test_loader  = DataLoader(test_dataset,  batch_size=256, shuffle=False)

images, labels = next(iter(train_loader))
print(f'Forme du tenseur images : {images.shape}')

**Questions :**

- Que représentent les quatre dimensions du tenseur `images` ?
  - `dim 0` : taille du lot (`batch_size = 64`)
  - `dim 1` : canaux (`1` → niveaux de gris)
  - `dim 2` : hauteur (28 pixels)
  - `dim 3` : largeur (28 pixels)

- Pourquoi mélanger les données avec `shuffle=True` ?
  Cela évite que le modèle apprenne l'ordre des classes, améliore la généralisation et stabilise le gradient stochastique.

## 2. Visualiser quelques images

In [ ]:
fig, axes = plt.subplots(1, 6, figsize=(12, 3))
fig.suptitle("Quelques images MNIST", fontsize=13)

for i, ax in enumerate(axes):
    # Annuler la normalisation pour l'affichage
    image = images[i, 0] * 0.3081 + 0.1307
    ax.imshow(image.numpy(), cmap='gray', vmin=0, vmax=1)
    ax.set_title(f'Classe : {labels[i].item()}')
    ax.axis('off')

plt.tight_layout()
plt.show()

## 3. Construire le CNN

Architecture *(entrée : 1 × 28 × 28)* :

| Couche | Sortie |
|--------|--------|
| Conv2d(1→16, 3×3, pad=1) + ReLU + MaxPool2d(2) | 16 × 14 × 14 |
| Conv2d(16→32, 3×3, pad=1) + ReLU + MaxPool2d(2) | 32 × 7 × 7 |
| Flatten | 1 568 |
| Linear(1568, 128) + ReLU + Dropout(0.3) | 128 |
| Linear(128, 10) — logits bruts | 10 |

> **Pourquoi 32 × 7 × 7 ?**  
> Après deux `MaxPool2d(2)` sur une entrée 28×28 : 28 → 14 → 7.  
> La seconde couche conv produit 32 cartes → 32 × 7 × 7 = **1 568** neurones.

> ⚠️ Ne pas ajouter de `Softmax` : `CrossEntropyLoss` attend les logits bruts.

In [ ]:
class MNISTClassifier(nn.Module):
    def __init__(self):
        super().__init__()

        self.features = nn.Sequential(
            # Bloc convolutif 1
            nn.Conv2d(in_channels=1, out_channels=16, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=2),   # 28x28 -> 14x14

            # Bloc convolutif 2
            nn.Conv2d(in_channels=16, out_channels=32, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=2)    # 14x14 -> 7x7
        )

        self.classifier = nn.Sequential(
            nn.Flatten(),                   # 32 x 7 x 7 = 1568
            nn.Linear(32 * 7 * 7, 128),
            nn.ReLU(),
            nn.Dropout(p=0.3),              # régularisation (question bonus)
            nn.Linear(128, 10)              # 10 classes, logits bruts
        )

    def forward(self, x):
        x = self.features(x)
        x = self.classifier(x)
        return x


model = MNISTClassifier().to(device)
print(model)
total_params = sum(p.numel() for p in model.parameters())
print(f'\nNombre total de paramètres : {total_params:,}')

## 4. Entraîner le réseau

In [ ]:
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

EPOCHS = 5
loss_history = []

for epoch in range(EPOCHS):
    model.train()
    total_loss = 0.0

    for batch_images, batch_labels in train_loader:
        batch_images = batch_images.to(device)
        batch_labels = batch_labels.to(device)

        # 1. Réinitialiser les gradients
        optimizer.zero_grad()

        # 2. Calculer les scores (logits)
        logits = model(batch_images)

        # 3. Calculer la perte
        loss = criterion(logits, batch_labels)

        # 4. Calculer les gradients (rétropropagation)
        loss.backward()

        # 5. Mettre à jour les paramètres
        optimizer.step()

        total_loss += loss.item() * batch_images.size(0)

    average_loss = total_loss / len(train_dataset)
    loss_history.append(average_loss)
    print(f'Époque {epoch + 1}/{EPOCHS} | Perte moyenne : {average_loss:.4f}')

In [ ]:
# Courbe de perte
plt.figure(figsize=(7, 4))
plt.plot(range(1, EPOCHS + 1), loss_history, marker='o', linewidth=2, color='royalblue')
plt.xlabel('Époque')
plt.ylabel('Perte (Cross-Entropy)')
plt.title("Courbe d'apprentissage — CNN MNIST")
plt.grid(True, linestyle='--', alpha=0.5)
plt.tight_layout()
plt.show()

## 5. Évaluer sur le jeu de test

In [ ]:
model.eval()

all_preds  = []
all_labels_list = []
correct = 0
total   = 0

with torch.no_grad():
    for batch_images, batch_labels in test_loader:
        batch_images = batch_images.to(device)
        batch_labels = batch_labels.to(device)

        logits = model(batch_images)

        # Classe ayant le score le plus élevé
        predictions = logits.argmax(dim=1)

        # Compter les bonnes prédictions
        correct += (predictions == batch_labels).sum().item()
        total   += batch_labels.size(0)

        all_preds.append(predictions.cpu())
        all_labels_list.append(batch_labels.cpu())

accuracy = correct / total
print(f'Accuracy sur le jeu de test : {accuracy:.2%}')

all_preds  = torch.cat(all_preds)
all_labels = torch.cat(all_labels_list)

## 6. Métriques : Precision, Recall, F1

In [ ]:
num_classes = 10
tp = torch.zeros(num_classes)
fp = torch.zeros(num_classes)
fn = torch.zeros(num_classes)

for c in range(num_classes):
    tp[c] = ((all_preds == c) & (all_labels == c)).sum().float()
    fp[c] = ((all_preds == c) & (all_labels != c)).sum().float()
    fn[c] = ((all_preds != c) & (all_labels == c)).sum().float()

precision_per_class = tp / (tp + fp + 1e-8)
recall_per_class    = tp / (tp + fn + 1e-8)
f1_per_class        = 2 * precision_per_class * recall_per_class / (
                        precision_per_class + recall_per_class + 1e-8)

macro_precision = precision_per_class.mean().item()
macro_recall    = recall_per_class.mean().item()
macro_f1        = f1_per_class.mean().item()

print(f"{'Classe':>7} {'Précision':>10} {'Rappel':>10} {'F1':>10}")
for c in range(num_classes):
    print(f'  {c:>5}   {precision_per_class[c]:.4f}     {recall_per_class[c]:.4f}     {f1_per_class[c]:.4f}')

print(f'\n--- Métriques macro ---')
print(f'  Précision : {macro_precision:.4f}')
print(f'  Rappel    : {macro_recall:.4f}')
print(f'  F1-score  : {macro_f1:.4f}')

## 7. Matrice de confusion

In [ ]:
confusion = torch.zeros(num_classes, num_classes, dtype=torch.long)
for true, pred in zip(all_labels, all_preds):
    confusion[true, pred] += 1

fig, ax = plt.subplots(figsize=(9, 7))
im = ax.imshow(confusion.numpy(), cmap='Blues')
plt.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
ax.set_xticks(range(num_classes))
ax.set_yticks(range(num_classes))
ax.set_xlabel('Classe prédite', fontsize=12)
ax.set_ylabel('Vraie classe',   fontsize=12)
ax.set_title('Matrice de confusion — MNIST CNN', fontsize=14)

threshold = confusion.max().item() / 2
for i in range(num_classes):
    for j in range(num_classes):
        val = confusion[i, j].item()
        color = 'white' if val > threshold else 'black'
        ax.text(j, i, str(val), ha='center', va='center', color=color, fontsize=9)

plt.tight_layout()
plt.show()

## 8. Afficher 10 images mal classées

In [ ]:
wrong_mask    = (all_preds != all_labels)
wrong_indices = wrong_mask.nonzero(as_tuple=False).squeeze()

# Récupérer toutes les images brutes du jeu de test
all_images_test = []
with torch.no_grad():
    for batch_images, _ in test_loader:
        all_images_test.append(batch_images)
all_images_test = torch.cat(all_images_test)   # (10000, 1, 28, 28)

fig, axes = plt.subplots(2, 5, figsize=(13, 6))
fig.suptitle('10 images mal classées', fontsize=14)

for ax, idx in zip(axes.ravel(), wrong_indices[:10]):
    idx = idx.item()
    img = all_images_test[idx, 0] * 0.3081 + 0.1307   # dénormalisation
    ax.imshow(img.numpy(), cmap='gray', vmin=0, vmax=1)
    true_lbl = all_labels[idx].item()
    pred_lbl = all_preds[idx].item()
    ax.set_title(f'Vrai : {true_lbl}  |  Prédit : {pred_lbl}', fontsize=9)
    ax.axis('off')

plt.tight_layout()
plt.show()
print(f'Nombre total d erreurs : {wrong_mask.sum().item()} / {len(all_labels)}')

## 9. Sauvegarde et rechargement du modèle

In [ ]:
SAVE_PATH = 'mnist_cnn.pth'
torch.save(model.state_dict(), SAVE_PATH)
print(f'Modèle sauvegardé → {SAVE_PATH}')

# Rechargement
model_loaded = MNISTClassifier().to(device)
model_loaded.load_state_dict(torch.load(SAVE_PATH, map_location=device, weights_only=True))
model_loaded.eval()

correct_reload = 0
with torch.no_grad():
    for batch_images, batch_labels in test_loader:
        batch_images = batch_images.to(device)
        batch_labels = batch_labels.to(device)
        logits = model_loaded(batch_images)
        preds  = logits.argmax(dim=1)
        correct_reload += (preds == batch_labels).sum().item()

accuracy_reload = correct_reload / len(test_dataset)
print(f'Accuracy après rechargement : {accuracy_reload:.2%}  (doit être identique)')
print('\n✅ Exercice 4 terminé !')